Foxf2 expression per sample (knockout check).

In [ ]:
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

plt.rcParams["font.family"] = "Arial" if "Arial" in [f.name for f in fm.fontManager.ttflist] else "sans-serif"

In [ ]:
method_path = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark") / "analysis/foxf2/Cellpose_1_nuclei_model"
adata = sc.read_h5ad(method_path / "adatas/adata_integrated.h5ad.gz")
adata.obs["cell_type"] = adata.obs["cell_type_revised"].astype(str).replace(
    {"Tanycytes": "Ependymal", "Astroependymal": "Astrocytes", "Neurons-Glyc-Gaba": "Neurons-Other"}
).astype("category")
order = ["WT", "ECKO", "PCKO", "GLKO"]
s = adata.obs["sample"].astype(str) + "_" + adata.obs["genotype"].astype(str)
adata.obs["sample2"] = pd.Categorical(s, categories=sorted(s.unique(), key=lambda x: order.index(x.rsplit("_", 1)[1])), ordered=True)
adata.obs["cell_type"].value_counts()

In [ ]:
for title, a in {
    "All cells": adata,
    "ECs only": adata[adata.obs["cell_type"] == "ECs"],
    "Pericytes only": adata[adata.obs["cell_type"] == "Pericytes"],
}.items():
    sc.pl.dotplot(a, ["Foxf2"], groupby="sample2", layer="volume_log1p_norm", vmin=0, vmax=0.75, dot_min=0, dot_max=1, figsize=(2, 4), title=title)

Foxf2 in ECs, WT vs ECKO:

In [ ]:
df = sc.get.obs_df(adata[adata.obs["genotype"].isin(["WT", "ECKO"]) & (adata.obs["cell_type"] == "ECs")], ["Foxf2", "genotype"])
df["genotype"] = df["genotype"].astype(str)
zero = df["Foxf2"].eq(0).groupby(df["genotype"]).mean()
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, d, title in [(axes[0], df, "All ECs"), (axes[1], df[df["Foxf2"] > 0], "ECs with Foxf2 > 0")]:
    sns.violinplot(data=d, x="genotype", y="Foxf2", inner="quart", cut=0, linewidth=1, ax=ax)
    sns.stripplot(data=d, x="genotype", y="Foxf2", jitter=0.2, size=1.5, alpha=0.2, color="black", ax=ax)
    ax.set_title(title)
axes[1].set_xlabel("\n".join(f"{g}: {p:.1%} with 0 counts" for g, p in zero.items()))
plt.tight_layout()
plt.show()